# Day 2 - Lab 1: Statistical analysis and patterns

**Goal:** describe the cleaned data properly, and decide which patterns are real. Yesterday you *found* that digital channels resolve faster. Today you ask: how confident are we, and what else is going on?

You are working on the cleaned dataset from Day 1. If you did not save it, this notebook rebuilds it for you.

## 1. Load the cleaned data
The loader reuses your Day 1 output, or rebuilds it from the raw file so you are never blocked.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

def find_data(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / 'data').is_dir():
            return p / 'data'
    raise FileNotFoundError('data/ folder not found')

DATA = find_data()

def _rebuild_clean():
    """Re-run the Day 1 cleaning from raw, so Day 2 is self-contained."""
    df = pd.read_csv(DATA / 'raw' / 'service_requests_raw.csv', dtype=str).drop_duplicates()
    df['priority'] = df['priority'].str.strip().str.title().replace({'2': 'Medium'})
    df['resolution_hours'] = pd.to_numeric(df['resolution_hours'], errors='coerce')
    iso   = pd.to_datetime(df['submitted_at'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
    named = pd.to_datetime(df['submitted_at'], format='%d-%b-%Y %H:%M', errors='coerce')
    df['submitted_at'] = iso.fillna(named)
    df['resolved_at'] = pd.to_datetime(df['resolved_at'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
    df.loc[(df['resolution_hours'] < 0) | (df['resolution_hours'] > 8760), 'resolution_hours'] = np.nan
    df.loc[(df['resolved_at'] < df['submitted_at']).fillna(False), 'resolved_at'] = pd.NaT
    df['citizen_age_band'] = df['citizen_age_band'].replace({'': pd.NA, 'Unknown': pd.NA}).fillna('Unknown')
    df['service_id'] = pd.to_numeric(df['service_id'], errors='coerce')
    svc = pd.read_csv(DATA / 'seeds' / 'services.csv')
    df = df.merge(svc[['service_id', 'target_resolution_hours']], on='service_id', how='left')
    resolved = df['status'].isin(['Resolved', 'Reopened']) & df['resolution_hours'].notna()
    df['sla_met'] = pd.NA
    df.loc[resolved, 'sla_met'] = (df.loc[resolved, 'resolution_hours'] <= df.loc[resolved, 'target_resolution_hours']).astype('int')
    return df

def load_clean():
    p = DATA / 'processed' / 'service_requests_clean.csv'
    if p.exists():
        print('Loaded cleaned dataset from Day 1:', p.name)
        return pd.read_csv(p, parse_dates=['submitted_at', 'resolved_at'])
    print('Cleaned file not found, rebuilding from raw (Day 1 cleaning)...')
    return _rebuild_clean()

df = load_clean()
channels = pd.read_csv(DATA / 'seeds' / 'channels.csv')
districts = pd.read_csv(DATA / 'seeds' / 'districts.csv')
df = df.merge(channels, on='channel_id', how='left')
print('shape:', df.shape)
df.head(3)

## 2. Shape of a single variable
`describe()` gives the five-number summary. But one number matters most here: the **skew**.
A long right tail means the mean is pulled upward and no longer represents a typical case.

In [ ]:
# TODO: your code here

In [ ]:
# TODO: your code here

A skew of **3.66** is strongly right-skewed. The mean (67.8 h) sits well above the median (44.1 h): a minority of very slow requests drag the average up.

**Decision:** for a skewed variable, report the **median** and the IQR, not the mean and standard deviation. Say *typical*, not *average*.

In [ ]:
# TODO: your code here

## 3. Group statistics
Split the target by the groups you care about. A `groupby` with several aggregations at once is the workhorse.

In [ ]:
# TODO: your code here

## 4. Patterns worth a manager's attention
Move from *how long* to *did we meet the promise*. `sla_met` is 1 (met), 0 (missed) or missing (still open).

In [ ]:
# SLA-met rate by channel (mean of a 0/1 column is the proportion of 1s)
# TODO: your code here

In [ ]:
# ... and by priority
# TODO: your code here

Two patterns stand out:
- Digital channels meet SLA around **59 to 61%** of the time; the call centre and walk-in centre only around **34 to 36%**.
- **High** priority requests meet SLA **89%** of the time, but **Low** priority only **38%**. Low-priority work is quietly left to slip.

The second pattern is the more interesting business question. Hold it for Lab 3.

## 5. Is the digital difference real, or luck?
The medians differ, but medians always differ *a bit*. A hypothesis test asks whether a gap this large could plausibly be noise.

Resolution time is heavily skewed, so we use the **Mann-Whitney U** test (it compares distributions without assuming a bell curve) rather than a plain t-test.

In [ ]:
# TODO: your code here

`p = 1.15e-64` is astronomically small: far below the usual 0.05 threshold. A gap this size, on samples this large, is not chance.

> **But statistical significance is not the same as importance.** With 12,000 rows, even a trivial difference would be 'significant'. Always pair the p-value with the effect size you can explain to a manager: *here, digital requests are resolved about 15 hours faster at the median*. The size is what matters, the p-value only rules out luck.

## Your turn
Answer these in the scratch cell.
1. Compute the SLA-met rate by `is_digital`. How large is the gap in percentage points?
2. Which single service (`service_id`) has the worst SLA-met rate? Is it a fair comparison?
3. Is `satisfaction_score` related to whether the SLA was met? Compare mean satisfaction for met vs missed.

In [ ]:
# your turn